[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amistein/ml-study-group/blob/main/notebooks/lesson_005.ipynb)

### Learning Rate

In the last lesson, we successfully ran gradient descent and found the optimal weight for price per sqft in a relatively few amount of steps. We calculated the numerical gradient of the loss function at the current weight values, multiplied that by a **learning rate**, and subtracted that from the current weight, to get the new weight. We simply chose an arbitrary small value as our learning rate. But in practice, the learning rate is one of the most important choices you make when training a model, and getting it wrong can be the difference between a model that learns (successfully figures out the optimal weight value) and one that doesn't. So let's explore what happens when we don't choose a good one.

First, let's review and slightly clarify from last lesson. The goal of our model training process is to find the point where our loss function output is lowest. It's like walking in a valley with the goal of getting to the lowest point. The gradient tells us in which direction down is, and how steep the hill is at our current point. The learning rate tells us how big of a step to take. If you take too small a step, you'll be walking for a long time. If you take too big a step, you might overshoot the bottom and end up on the other side of the valley, maybe even higher up than where you started. In model training, overshooting our minimum is called **divergence**. The weight bounces back and forth and never reaches the minimum. (This is the opposite of **convergence**, which is what we want).

<img src="../images/learning_rate.png" width="500">

[image credit](https://mohitmishra786687.medium.com/the-learning-rate-a-hyperparameter-that-matters-b2f3b68324ab)

The tricky part is that there's no universal "correct" learning rate. It depends on the specific loss function, the scale of the data, and the number of weights. A learning rate of `0.01` might be perfect for one problem and cause another to diverge. In practice, people often try a few values and see which one produces the smoothest, fastest convergence. There are more sophisticated strategies for setting and adjusting the learning rate during training, but for now, trying a few values and looking at convergence curves is a good starting point, and that's what we'll be doing in the exercises.

### **Exercises**
#### Implement the following functions, and ensure the tests pass.

*Reading through the test cases is a good way to make sure you understand what your code is doing.*

First, let's bring back the building blocks from lesson 4: the loss function, numerical gradient, and gradient descent.

In [ ]:
# --- Setup (do not modify) ---
sqft = [800, 1200, 1500, 2000, 2500]
actual_prices = [150000, 210000, 260000, 340000, 420000]

def mse(weight):
    """Mean squared error for our housing data, as a function of price_per_sqft."""
    predictions = [weight * s for s in sqft]
    return sum((predictions[i] - actual_prices[i]) ** 2 for i in range(len(actual_prices))) / len(actual_prices)

def numerical_gradient(f, x, h=0.001):
    """Compute the slope of function f at point x."""
    return (f(x + h) - f(x)) / h

def gradient_descent(f, initial_weight, learning_rate, num_steps):
    """Run gradient descent to minimize function f. Returns the list of weights at each step."""
    weights = [initial_weight]
    weight = initial_weight
    for _ in range(num_steps):
        grad = numerical_gradient(f, weight)
        weight = weight - learning_rate * grad
        weights.append(weight)
    return weights

Now run gradient descent three times with different learning rates and observe the behavior of each. For each learning rate, compute the list of weights and the corresponding list of losses at each step.

In [ ]:
def run_experiments(f, learning_rates, initial_weight, num_steps):
    """Run gradient descent with each learning rate and collect the losses.

    Args:
        f: the loss function
        learning_rates: dict mapping label -> learning rate value
        initial_weight: starting weight for all experiments
        num_steps: number of gradient descent steps
    Returns:
        dict mapping label -> list of losses at each step
        (each list has length num_steps + 1, including the initial loss)
    """
    # YOUR CODE HERE
    # Hint: for each label and learning rate, call gradient_descent,
    # then compute the loss at each weight in the returned list.
    raise NotImplementedError()


# --- Tests (do not modify) ---
learning_rates = {
    "too_small": 1e-8,
    "good":      1e-7,
    "too_big":   4e-7,
}
initial_weight=50
num_steps = 30
_results = run_experiments(mse, learning_rates, initial_weight, num_steps)

# Check structure
assert set(_results.keys()) == {"too_small", "good", "too_big"}, f"Expected 3 keys, got {_results.keys()}"
for _label in _results:
    assert len(_results[_label]) == 31, f"Expected 31 losses for {_label}, got {len(_results[_label])}"

# All three start from the same initial loss
assert _results["too_small"][0] == _results["good"][0] == _results["too_big"][0], "All should start at the same loss"

# Too small: loss decreased, but not nearly as much as the good rate
assert _results["too_small"][-1] < _results["too_small"][0], "Too-small should still reduce loss"
assert _results["too_small"][-1] > _results["good"][-1] * 10, "Too-small should be much worse than good"

# Good: converged to a low loss
assert _results["good"][-1] < 100_000_000, f"Good should converge to low loss, got {_results['good'][-1]:,.0f}"

# Too big: diverged (loss went UP, not down)
assert _results["too_big"][-1] > _results["too_big"][0], "Too-big should diverge: final loss > initial loss"

print(f"Too small (lr=1e-8):  final loss = {_results['too_small'][-1]:,.0f}")
print(f"Good      (lr=1e-7):  final loss = {_results['good'][-1]:,.0f}")
print(f"Too big   (lr=4e-7):  final loss = {_results['too_big'][-1]:.2e}")
print("✅ All tests passed!")

Now let's visualize the difference. Plot the loss at each step for all three learning rates on the same chart. This is the standard way to compare training runs: put the step number on the x-axis, the loss on the y-axis, and plot one line per experiment.

In [ ]:
from matplotlib import pyplot as plt

# YOUR CODE HERE: plot the loss curves for all three learning rates.
# Use plt.plot(range(31), losses, label=label) for each.
# Hint: loop over _results.items()



# --- Plot formatting (do not modify) ---
# Cap the y-axis so the diverging run doesn't squash the other two curves.
# This is a common trick when comparing training runs with very different scales.
plt.ylim(0, _results["too_small"][0] * 1.2)
plt.xlabel("Step")
plt.ylabel("MSE Loss")
plt.title("Gradient Descent: Effect of Learning Rate")
plt.legend()
plt.show()

The "too big" line shoots off the top of the chart almost immediately, which is why we capped the y-axis. Without that cap, the diverging run's loss is so big that the other two curves would be squashed flat against the bottom. But even with the cap, you can see the key difference: the "good" learning rate drops to the minimum in about 5 steps, while the "too small" one is still moving downward after 30 and is still not as good. In real training scenarios with millions of weights and large datasets, that difference translates directly into hours or days of wasted compute time.